In [ ]:
"""
PBM Claims Rejection EDA — Full Dataset (10 lakh rows)
=======================================================
Run this script on your local machine where the dataset lives.
Generates a folder: pbm_eda_output/  with all charts as PNGs
and a summary CSV for each section.

Usage:
    python pbm_eda_full.py --file "your_data.xlsx"
    python pbm_eda_full.py --file "your_data.csv"

Requirements:
    pip install pandas matplotlib seaborn openpyxl
"""

import argparse
import os
import re
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings("ignore")

# ── output folder ─────────────────────────────────────────────────────────────
OUT = "pbm_eda_output"
os.makedirs(OUT, exist_ok=True)

# ── chart style ───────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})
REJECT_COLOR  = "#E24B4A"
APPROVE_COLOR = "#639922"
PALETTE       = [REJECT_COLOR, APPROVE_COLOR]


# ══════════════════════════════════════════════════════════════════════════════
# 0. LOAD DATA
# ══════════════════════════════════════════════════════════════════════════════

def load_data(filepath: str) -> pd.DataFrame:
    ext = filepath.rsplit(".", 1)[-1].lower()
    print(f"[load] reading {filepath} ...")
    if ext in ("xlsx", "xlsm", "xls"):
        df = pd.read_excel(filepath)
    elif ext == "csv":
        df = pd.read_csv(filepath, low_memory=False)
    else:
        raise ValueError(f"Unsupported file type: {ext}")
    print(f"[load] shape: {df.shape}")
    return df



In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# 1. PREPROCESSING
# ══════════════════════════════════════════════════════════════════════════════

def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    print("[prep] preprocessing ...")

    # ── target: binary rejection flag ─────────────────────────────────────────
    df["IS_REJECTED"] = (df["TREATMENT_APPR_STS"] == "QLM_REJECT").astype(int)

    # ── date parsing ──────────────────────────────────────────────────────────
    if "SERVICE_DT" in df.columns:
        df["SERVICE_DT"] = pd.to_datetime(df["SERVICE_DT"], errors="coerce")
        df["SERVICE_YEAR"]  = df["SERVICE_DT"].dt.year
        df["SERVICE_MONTH"] = df["SERVICE_DT"].dt.month
        df["SERVICE_MONTH_NAME"] = df["SERVICE_DT"].dt.strftime("%b %Y")
        df["SERVICE_DOW"] = df["SERVICE_DT"].dt.day_name()

    # ── DRUG_DURATION → days (numeric) ────────────────────────────────────────
    if "DRUG_DURATION" in df.columns:
        df["DRUG_DURATION_DAYS"] = df["DRUG_DURATION"].apply(_parse_duration)

    # ── age groups (if MEM_AGE exists) ────────────────────────────────────────
    if "MEM_AGE" in df.columns:
        df["AGE_GROUP"] = pd.cut(
            df["MEM_AGE"],
            bins=[0, 18, 30, 45, 60, 200],
            labels=["0-18", "19-30", "31-45", "46-60", "60+"],
        )

    # ── drug-diagnosis combo key ──────────────────────────────────────────────
    if "DRUG_CODE" in df.columns and "PA_PRIMARY_DIAG" in df.columns:
        df["DRUG_DIAG_COMBO"] = (
            df["DRUG_CODE"].fillna("NO_DRUG").str.strip()
            + " | "
            + df["PA_PRIMARY_DIAG"].fillna("NO_DIAG").str.strip()
        )

    # ── derived rate features (using full dataset history) ────────────────────
    # These replace the columns that were missing in your small extract.
    # DRUG_REJ_RATE: % of times a drug was rejected across all claims
    if "DRUG_CODE" in df.columns:
        drug_stats = df.groupby("DRUG_CODE")["IS_REJECTED"].mean().rename("DRUG_REJ_RATE")
        df = df.join(drug_stats, on="DRUG_CODE")

    # DIAG_REJ_RATE: % of times a diagnosis code was rejected
    if "PA_PRIMARY_DIAG" in df.columns:
        diag_stats = df.groupby("PA_PRIMARY_DIAG")["IS_REJECTED"].mean().rename("DIAG_REJ_RATE")
        df = df.join(diag_stats, on="PA_PRIMARY_DIAG")

    # DOC_REJ_RATE: % of doctor's claims that were rejected
    if "DOC_LIC_NO" in df.columns:
        doc_stats = df.groupby("DOC_LIC_NO")["IS_REJECTED"].mean().rename("DOC_REJ_RATE")
        df = df.join(doc_stats, on="DOC_LIC_NO")

    print(f"[prep] done. columns: {len(df.columns)}")
    return df


def _parse_duration(val) -> float:
    """Convert '60D', '3M', '2W', '1Y' etc. to days as float."""
    if pd.isna(val):
        return float("nan")
    val = str(val).strip().upper()
    m = re.match(r"(\d+(?:\.\d+)?)\s*([DWMY])", val)
    if not m:
        return float("nan")
    n, unit = float(m.group(1)), m.group(2)
    return {"D": 1, "W": 7, "M": 30, "Y": 365}[unit] * n


# ══════════════════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def #savefig(name: str):
    path = os.path.join(OUT, name)
    plt.#savefig(path, bbox_inches="tight")
    plt.close()
    print(f"  [saved] {path}")


def rejection_rate_bar(
    series: pd.Series,
    title: str,
    filename: str,
    top_n: int = 20,
    color: str = REJECT_COLOR,
):
    """Simple horizontal bar of rejection rates (%)."""
    data = series.sort_values(ascending=True).tail(top_n)
    fig, ax = plt.subplots(figsize=(10, max(4, len(data) * 0.45)))
    bars = ax.barh(data.index.astype(str), data.values * 100, color=color, alpha=0.85)
    ax.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=9)
    ax.set_xlabel("Rejection rate (%)")
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlim(0, 115)
    #savefig(filename)


def stacked_count_bar(
    df: pd.DataFrame,
    col: str,
    title: str,
    filename: str,
    top_n: int = 20,
):
    """Stacked bar: approved vs rejected counts."""
    grp = (
        df.groupby([col, "TREATMENT_APPR_STS"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=["QLM_REJECT", "QLM_APPROVED"], fill_value=0)
    )
    grp["total"] = grp.sum(axis=1)
    grp = grp.nlargest(top_n, "total").drop(columns="total")
    fig, ax = plt.subplots(figsize=(12, max(5, len(grp) * 0.5)))
    grp.plot(kind="barh", stacked=True, ax=ax,
             color=[REJECT_COLOR, APPROVE_COLOR], alpha=0.85)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Number of claims")
    ax.legend(["Rejected", "Approved"], loc="lower right")
    #savefig(filename)



In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# 2. OVERVIEW
# ══════════════════════════════════════════════════════════════════════════════

def section_overview(df: pd.DataFrame):
    print("\n[section] OVERVIEW")
    total = len(df)
    rejected = df["IS_REJECTED"].sum()
    approved = total - rejected

    summary = {
        "Total claims": total,
        "Rejected (QLM_REJECT)": rejected,
        "Approved (QLM_APPROVED)": approved,
        "Overall rejection rate (%)": round(rejected / total * 100, 2),
        "Avg TREAT_EST_AMT": round(df["TREAT_EST_AMT"].mean(), 2) if "TREAT_EST_AMT" in df.columns else "N/A",
        "Avg TREAT_REJ_AMT": round(df["TREAT_REJ_AMT"].mean(), 2) if "TREAT_REJ_AMT" in df.columns else "N/A",
        "Total TREAT_REJ_AMT": df["TREAT_REJ_AMT"].sum() if "TREAT_REJ_AMT" in df.columns else "N/A",
    }
    pd.Series(summary).to_csv(os.path.join(OUT, "00_overview_summary.csv"), header=["value"])
    for k, v in summary.items():
        print(f"   {k}: {v}")

    # pie chart
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.pie(
        [rejected, approved],
        labels=[f"Rejected\n{rejected:,}", f"Approved\n{approved:,}"],
        colors=[REJECT_COLOR, APPROVE_COLOR],
        autopct="%1.1f%%",
        startangle=140,
    )
    ax.set_title("Claim outcome distribution", fontsize=13, fontweight="bold")
    # #savefig("00_outcome_pie.png")



In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# 3. AGE ANALYSIS  (only runs if MEM_AGE exists)
# ══════════════════════════════════════════════════════════════════════════════

def section_age(df: pd.DataFrame):
    if "MEM_AGE" not in df.columns:
        print("\n[section] AGE — skipped (MEM_AGE not in dataset)")
        return
    print("\n[section] AGE")

    # histogram
    fig, ax = plt.subplots(figsize=(10, 5))
    df["MEM_AGE"].hist(bins=40, color="#378ADD", alpha=0.85, ax=ax)
    ax.set_title("Patient age distribution", fontsize=13, fontweight="bold")
    ax.set_xlabel("Age")
    ax.set_ylabel("Count")
    # #savefig("01_age_histogram.png")

    # boxplot: age vs outcome
    fig, ax = plt.subplots(figsize=(7, 5))
    df.boxplot(column="MEM_AGE", by="TREATMENT_APPR_STS", ax=ax,
               boxprops=dict(color="steelblue"),
               medianprops=dict(color="red"),
               flierprops=dict(marker="o", markersize=2, alpha=0.3))
    ax.set_title("Age by claim outcome", fontsize=13, fontweight="bold")
    ax.set_xlabel("Outcome")
    ax.set_ylabel("Age")
    plt.suptitle("")
    # #savefig("01_age_boxplot.png")

    # rejection rate by age group
    if "AGE_GROUP" in df.columns:
        rate = df.groupby("AGE_GROUP", observed=True)["IS_REJECTED"].mean()
        fig, ax = plt.subplots(figsize=(8, 4))
        rate.mul(100).plot(kind="bar", color=REJECT_COLOR, alpha=0.85, ax=ax)
        ax.set_title("Rejection rate by age group", fontsize=13, fontweight="bold")
        ax.set_ylabel("Rejection rate (%)")
        ax.set_xlabel("Age group")
        plt.xticks(rotation=0)
        # #savefig("01_age_group_rejection_rate.png")

        rate.to_csv(os.path.join(OUT, "01_age_group_rejection_rate.csv"))



In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# 4. GENDER ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════

def section_gender(df: pd.DataFrame):
    if "MEM_GENDER" not in df.columns:
        print("\n[section] GENDER — skipped"); return
    print("\n[section] GENDER")

    stacked_count_bar(df, "MEM_GENDER", "Claims by gender and outcome", "02_gender_stacked.png", top_n=10)

    rate = df.groupby("MEM_GENDER")["IS_REJECTED"].agg(["mean", "sum", "count"])
    rate.columns = ["Rejection Rate", "Rejected", "Total"]
    rate["Rejection Rate (%)"] = (rate["Rejection Rate"] * 100).round(2)
    rate.to_csv(os.path.join(OUT, "02_gender_stats.csv"))
    print(rate)



In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# 5. DIAGNOSIS ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════

def section_diagnosis(df: pd.DataFrame):
    if "PA_PRIMARY_DIAG" not in df.columns:
        print("\n[section] DIAGNOSIS — skipped"); return
    print("\n[section] DIAGNOSIS")

    stacked_count_bar(df, "PA_PRIMARY_DIAG", "Top 20 diagnoses: approved vs rejected",
                      "03_diagnosis_stacked.png", top_n=20)

    rate = df.groupby("PA_PRIMARY_DIAG")["IS_REJECTED"].agg(["mean", "sum", "count"])
    rate.columns = ["Rej_Rate", "Rejected", "Total"]
    rate = rate[rate["Total"] >= 10]  # min 10 claims to be meaningful
    rejection_rate_bar(rate["Rej_Rate"],
                       "Top diagnoses by rejection rate (min 10 claims)",
                       "03_diagnosis_rejection_rate.png")
    rate.sort_values("Rej_Rate", ascending=False).to_csv(os.path.join(OUT, "03_diagnosis_rates.csv"))



In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# 6. DRUG ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════

def section_drug(df: pd.DataFrame):
    if "DRUG_CODE" not in df.columns:
        print("\n[section] DRUG — skipped"); return
    drug_df = df[df["DRUG_CODE"].notna()].copy()
    print(f"\n[section] DRUG  ({len(drug_df):,} rows with drug codes)")

    stacked_count_bar(drug_df, "DRUG_CODE", "Top 20 drug codes: approved vs rejected",
                      "04_drug_stacked.png", top_n=20)

    rate = drug_df.groupby("DRUG_CODE")["IS_REJECTED"].agg(["mean", "sum", "count"])
    rate.columns = ["Rej_Rate", "Rejected", "Total"]
    rate = rate[rate["Total"] >= 10]
    rejection_rate_bar(rate["Rej_Rate"],
                       "Top drugs by rejection rate (min 10 claims)",
                       "04_drug_rejection_rate.png")
    rate.sort_values("Rej_Rate", ascending=False).to_csv(os.path.join(OUT, "04_drug_rates.csv"))

    # drug duration analysis
    if "DRUG_DURATION_DAYS" in df.columns:
        fig, ax = plt.subplots(figsize=(10, 5))
        df.boxplot(column="DRUG_DURATION_DAYS", by="TREATMENT_APPR_STS", ax=ax,
                   boxprops=dict(color="steelblue"),
                   medianprops=dict(color="red"),
                   flierprops=dict(marker="o", markersize=1, alpha=0.2))
        ax.set_title("Drug duration (days) by outcome", fontsize=13, fontweight="bold")
        ax.set_xlabel("Outcome")
        ax.set_ylabel("Duration (days)")
        plt.suptitle("")
        # #savefig("04_drug_duration_boxplot.png")



In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# 7. DRUG + DIAGNOSIS COMBO
# ══════════════════════════════════════════════════════════════════════════════

def section_drug_diag_combo(df: pd.DataFrame):
    if "DRUG_DIAG_COMBO" not in df.columns:
        print("\n[section] DRUG-DIAG COMBO — skipped"); return
    drug_df = df[df["DRUG_CODE"].notna()].copy()
    print(f"\n[section] DRUG-DIAG COMBO")

    rate = drug_df.groupby("DRUG_DIAG_COMBO")["IS_REJECTED"].agg(["mean", "sum", "count"])
    rate.columns = ["Rej_Rate", "Rejected", "Total"]
    rate = rate[rate["Total"] >= 5]  # min 5 to be meaningful

    # top rejected combos
    top_rej = rate.sort_values("Rej_Rate", ascending=True).tail(20)
    fig, ax = plt.subplots(figsize=(12, max(5, len(top_rej) * 0.5)))
    ax.barh(top_rej.index.astype(str), top_rej["Rej_Rate"] * 100, color=REJECT_COLOR, alpha=0.85)
    ax.set_title("Top drug-diagnosis combos by rejection rate (min 5 claims)",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Rejection rate (%)")
    #savefig("05_drug_diag_combo_rejected.png")

    # top approved combos
    top_appr = rate.sort_values("Rej_Rate", ascending=True).head(20)
    fig, ax = plt.subplots(figsize=(12, max(5, len(top_appr) * 0.5)))
    ax.barh(top_appr.index.astype(str), (1 - top_appr["Rej_Rate"]) * 100, color=APPROVE_COLOR, alpha=0.85)
    ax.set_title("Top drug-diagnosis combos by approval rate (min 5 claims)",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Approval rate (%)")
    #savefig("05_drug_diag_combo_approved.png")

    rate.sort_values("Rej_Rate", ascending=False).to_csv(os.path.join(OUT, "05_drug_diag_combo_rates.csv"))



In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# 8. SERVICE TYPE
# ══════════════════════════════════════════════════════════════════════════════

def section_service_type(df: pd.DataFrame):
    if "SERVICE_TYPE" not in df.columns:
        print("\n[section] SERVICE TYPE — skipped"); return
    print("\n[section] SERVICE TYPE")

    stacked_count_bar(df, "SERVICE_TYPE", "Claims by service type (I=Inpatient, O=Outpatient)",
                      "06_service_type_stacked.png", top_n=10)

    rate = df.groupby("SERVICE_TYPE")["IS_REJECTED"].agg(["mean", "sum", "count"])
    rate.columns = ["Rej_Rate", "Rejected", "Total"]
    rate["Rej_Rate (%)"] = (rate["Rej_Rate"] * 100).round(2)
    rate.to_csv(os.path.join(OUT, "06_service_type_stats.csv"))
    print(rate)


# ══════════════════════════════════════════════════════════════════════════════
# 9. REJECTION CATEGORY / REMARKS
# ══════════════════════════════════════════════════════════════════════════════

def section_rejection_codes(df: pd.DataFrame):
    print("\n[section] REJECTION CODES")

    for col, fname_prefix in [
        ("REJ_CODE", "07_rej_code"),
        ("REJ_REMARKS", "07_rej_remarks"),
        ("PBM_REJ_CODE", "07_pbm_rej_code"),
        ("PBM_REJ_DESC", "07_pbm_rej_desc"),
    ]:
        if col not in df.columns:
            continue
        rej_df = df[df[col].notna()].copy()
        vc = rej_df[col].value_counts().head(20)
        if vc.empty:
            continue

        # truncate long labels
        vc.index = [str(x)[:60] for x in vc.index]

        fig, ax = plt.subplots(figsize=(12, max(4, len(vc) * 0.45)))
        vc.sort_values().plot(kind="barh", color=REJECT_COLOR, alpha=0.85, ax=ax)
        ax.set_title(f"Top rejection reasons: {col}", fontsize=13, fontweight="bold")
        ax.set_xlabel("Count")
        #savefig(f"{fname_prefix}_top20.png")
        vc.to_csv(os.path.join(OUT, f"{fname_prefix}_top20.csv"))



In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# 10. DOCTOR ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════

def section_doctor(df: pd.DataFrame):
    if "DOC_LIC_NO" not in df.columns:
        print("\n[section] DOCTOR — skipped"); return
    print("\n[section] DOCTOR")

    stacked_count_bar(df, "DOC_LIC_NO", "Top 20 doctors by claim volume (approved vs rejected)",
                      "08_doctor_stacked.png", top_n=20)

    rate = df.groupby("DOC_LIC_NO")["IS_REJECTED"].agg(["mean", "sum", "count"])
    rate.columns = ["Rej_Rate", "Rejected", "Total"]
    rate = rate[rate["Total"] >= 10]
    rejection_rate_bar(rate["Rej_Rate"],
                       "Doctors with highest rejection rate (min 10 claims)",
                       "08_doctor_rejection_rate.png")
    rate.sort_values("Rej_Rate", ascending=False).to_csv(os.path.join(OUT, "08_doctor_rates.csv"))



In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# 11. FACILITY ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════

def section_facility(df: pd.DataFrame):
    if "FACILITIES" not in df.columns:
        print("\n[section] FACILITY — skipped"); return
    print("\n[section] FACILITY")

    stacked_count_bar(df, "FACILITIES", "Top 20 facilities: approved vs rejected",
                      "09_facility_stacked.png", top_n=20)

    rate = df.groupby("FACILITIES")["IS_REJECTED"].agg(["mean", "sum", "count"])
    rate.columns = ["Rej_Rate", "Rejected", "Total"]
    rate = rate[rate["Total"] >= 10]
    rejection_rate_bar(rate["Rej_Rate"],
                       "Facilities with highest rejection rate (min 10 claims)",
                       "09_facility_rejection_rate.png")
    rate.sort_values("Rej_Rate", ascending=False).to_csv(os.path.join(OUT, "09_facility_rates.csv"))



In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# 12. CORRELATION / NUMERICAL FEATURES
# ══════════════════════════════════════════════════════════════════════════════

def section_correlation(df: pd.DataFrame):
    print("\n[section] CORRELATION")

    num_cols = [c for c in [
        "MEM_AGE", "DRUG_DURATION_DAYS", "ADJUDICATION_SCORE", "FLAGGING_SCORE",
        "TREAT_EST_AMT", "TREAT_REJ_AMT", "TREAT_APPR_AMT", "PA_QTY",
        "DRUG_REJ_RATE", "DIAG_REJ_RATE", "DOC_REJ_RATE", "IS_REJECTED",
    ] if c in df.columns]

    if len(num_cols) < 3:
        print("  not enough numeric columns for correlation"); return

    corr = df[num_cols].corr()
    fig, ax = plt.subplots(figsize=(max(8, len(num_cols)), max(6, len(num_cols) - 1)))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlGn",
                center=0, linewidths=0.5, ax=ax,
                cbar_kws={"shrink": 0.7})
    ax.set_title("Correlation heatmap — numerical features", fontsize=13, fontweight="bold")
    #savefig("10_correlation_heatmap.png")
    corr.to_csv(os.path.join(OUT, "10_correlation_matrix.csv"))

    # IS_REJECTED correlations specifically
    if "IS_REJECTED" in corr.columns:
        rej_corr = corr["IS_REJECTED"].drop("IS_REJECTED").sort_values()
        fig, ax = plt.subplots(figsize=(8, max(4, len(rej_corr) * 0.5)))
        colors = [REJECT_COLOR if v > 0 else APPROVE_COLOR for v in rej_corr]
        ax.barh(rej_corr.index.astype(str), rej_corr.values, color=colors, alpha=0.85)
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_title("Feature correlation with IS_REJECTED", fontsize=13, fontweight="bold")
        ax.set_xlabel("Pearson correlation")
        #savefig("10_correlation_with_target.png")



In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# 13. OUTLIER ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════

def section_outliers(df: pd.DataFrame):
    print("\n[section] OUTLIERS")

    outlier_cols = [c for c in [
        "DRUG_DURATION_DAYS", "TREAT_EST_AMT", "TREAT_REJ_AMT",
        "ADJUDICATION_SCORE", "FLAGGING_SCORE", "MEM_AGE", "PA_QTY",
    ] if c in df.columns]

    if not outlier_cols:
        print("  no outlier columns found"); return

    # boxplots
    n = len(outlier_cols)
    fig, axes = plt.subplots(1, n, figsize=(n * 3.5, 5))
    if n == 1:
        axes = [axes]
    for ax, col in zip(axes, outlier_cols):
        df[col].dropna().plot(kind="box", ax=ax,
                              boxprops=dict(color="steelblue"),
                              medianprops=dict(color="red"),
                              flierprops=dict(marker="o", markersize=1, alpha=0.2))
        ax.set_title(col, fontsize=10)
    fig.suptitle("Outlier overview (box plots)", fontsize=13, fontweight="bold")
    #savefig("11_outlier_boxplots.png")

    # IQR summary
    records = []
    for col in outlier_cols:
        s = df[col].dropna()
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_out = ((s < lower) | (s > upper)).sum()
        records.append({"column": col, "Q1": q1, "Q3": q3, "IQR": iqr,
                        "lower_fence": lower, "upper_fence": upper,
                        "outlier_count": n_out, "outlier_pct": round(n_out / len(s) * 100, 2)})
    pd.DataFrame(records).to_csv(os.path.join(OUT, "11_outlier_iqr_summary.csv"), index=False)
    print(pd.DataFrame(records)[["column", "outlier_count", "outlier_pct"]])



In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# 14. TIME-BASED ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════

def section_time(df: pd.DataFrame):
    if "SERVICE_DT" not in df.columns:
        print("\n[section] TIME — skipped"); return
    print("\n[section] TIME")

    # monthly claims volume
    monthly = df.groupby("SERVICE_MONTH_NAME")["IS_REJECTED"].agg(["sum", "count"])
    monthly.columns = ["Rejected", "Total"]
    monthly["Approved"] = monthly["Total"] - monthly["Rejected"]
    monthly["Rej_Rate (%)"] = (monthly["Rejected"] / monthly["Total"] * 100).round(2)

    fig, ax = plt.subplots(figsize=(14, 5))
    monthly[["Rejected", "Approved"]].plot(kind="bar", stacked=True, ax=ax,
                                            color=[REJECT_COLOR, APPROVE_COLOR], alpha=0.85)
    ax.set_title("Monthly claims volume: approved vs rejected", fontsize=13, fontweight="bold")
    ax.set_xlabel("Month")
    ax.set_ylabel("Claims")
    plt.xticks(rotation=45, ha="right")
    #savefig("12_monthly_claims.png")
    monthly.to_csv(os.path.join(OUT, "12_monthly_claims.csv"))

    # rejection rate trend
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(monthly.index, monthly["Rej_Rate (%)"], marker="o",
            color=REJECT_COLOR, linewidth=2)
    ax.set_title("Monthly rejection rate trend (%)", fontsize=13, fontweight="bold")
    ax.set_ylabel("Rejection rate (%)")
    ax.set_xlabel("Month")
    plt.xticks(rotation=45, ha="right")
    ax.axhline(monthly["Rej_Rate (%)"].mean(), color="gray", linestyle="--",
               linewidth=1, label=f"Mean {monthly['Rej_Rate (%)'].mean():.1f}%")
    ax.legend()
    #savefig("12_monthly_rejection_trend.png")

    # day of week
    if "SERVICE_DOW" in df.columns:
        dow_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
        dow = df.groupby("SERVICE_DOW")["IS_REJECTED"].agg(["mean","count"])
        dow = dow.reindex(dow_order).dropna()
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.bar(dow.index, dow["mean"] * 100, color="#7F77DD", alpha=0.85)
        ax.set_title("Rejection rate by day of week", fontsize=13, fontweight="bold")
        ax.set_ylabel("Rejection rate (%)")
        plt.xticks(rotation=30, ha="right")
        #savefig("12_dow_rejection_rate.png")



In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# 15. ADJUDICATION & FLAGGING SCORES
# ══════════════════════════════════════════════════════════════════════════════

def section_scores(df: pd.DataFrame):
    print("\n[section] SCORES")

    for col in ["ADJUDICATION_SCORE", "FLAGGING_SCORE"]:
        if col not in df.columns:
            continue
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        # distribution by outcome
        for outcome, color in [("QLM_REJECT", REJECT_COLOR), ("QLM_APPROVED", APPROVE_COLOR)]:
            subset = df[df["TREATMENT_APPR_STS"] == outcome][col].dropna()
            axes[0].hist(subset, bins=30, alpha=0.6, color=color,
                         label=outcome.replace("QLM_", ""))
        axes[0].set_title(f"{col} distribution by outcome", fontsize=11, fontweight="bold")
        axes[0].set_xlabel(col)
        axes[0].legend()

        # boxplot by outcome
        df.boxplot(column=col, by="TREATMENT_APPR_STS", ax=axes[1],
                   boxprops=dict(color="steelblue"),
                   medianprops=dict(color="red"),
                   flierprops=dict(marker="o", markersize=1, alpha=0.2))
        axes[1].set_title(f"{col} by outcome", fontsize=11, fontweight="bold")
        axes[1].set_xlabel("Outcome")
        plt.suptitle("")
        #savefig(f"13_{col.lower()}_analysis.png")



In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# 16. INS_TREAT_DESC (treatment category)
# ══════════════════════════════════════════════════════════════════════════════

def section_treatment_desc(df: pd.DataFrame):
    if "INS_TREAT_DESC" not in df.columns:
        print("\n[section] INS_TREAT_DESC — skipped"); return
    print("\n[section] INS_TREAT_DESC")

    stacked_count_bar(df, "INS_TREAT_DESC",
                      "Top 20 treatment categories: approved vs rejected",
                      "14_ins_treat_desc_stacked.png", top_n=20)

    rate = df.groupby("INS_TREAT_DESC")["IS_REJECTED"].agg(["mean", "sum", "count"])
    rate.columns = ["Rej_Rate", "Rejected", "Total"]
    rate = rate[rate["Total"] >= 10]
    rejection_rate_bar(rate["Rej_Rate"],
                       "Treatment categories with highest rejection rate (min 10 claims)",
                       "14_ins_treat_desc_rejection_rate.png")
    rate.sort_values("Rej_Rate", ascending=False).to_csv(
        os.path.join(OUT, "14_ins_treat_desc_rates.csv"))



In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# 17. FEATURE IMPORTANCE PREVIEW (chi-squared)
# ══════════════════════════════════════════════════════════════════════════════

def section_feature_importance(df: pd.DataFrame):
    """Quick chi-squared / Cramér's V proxy to rank categorical features."""
    print("\n[section] FEATURE IMPORTANCE PREVIEW")
    try:
        from scipy.stats import chi2_contingency
    except ImportError:
        print("  scipy not installed, skipping. pip install scipy")
        return

    cat_cols = [c for c in [
        "MEM_GENDER", "SERVICE_TYPE", "FACILITIES", "DOC_LIC_NO",
        "PA_PRIMARY_DIAG", "DRUG_CODE", "INS_TREAT_DESC",
        "REJ_CODE", "PA_APPR_STATUS", "PBM_APPR_STS",
    ] if c in df.columns]

    records = []
    for col in cat_cols:
        try:
            ct = pd.crosstab(df[col], df["IS_REJECTED"])
            chi2, p, dof, _ = chi2_contingency(ct)
            n = ct.values.sum()
            v = (chi2 / (n * (min(ct.shape) - 1))) ** 0.5  # Cramér's V
            records.append({"feature": col, "cramers_v": round(v, 4), "chi2": round(chi2, 2), "p_value": p})
        except Exception:
            pass

    if not records:
        return

    result = pd.DataFrame(records).sort_values("cramers_v", ascending=False)
    result.to_csv(os.path.join(OUT, "15_feature_importance_cramers_v.csv"), index=False)

    fig, ax = plt.subplots(figsize=(9, max(4, len(result) * 0.5)))
    result_sorted = result.sort_values("cramers_v")
    ax.barh(result_sorted["feature"], result_sorted["cramers_v"],
            color="#7F77DD", alpha=0.85)
    ax.set_title("Categorical feature importance (Cramér's V vs IS_REJECTED)\n"
                 "Higher = stronger association with rejection",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("Cramér's V  (0 = no association, 1 = perfect)")
    #savefig("15_feature_importance_cramers_v.png")
    print(result.to_string(index=False))



In [ ]:

# parser = argparse.ArgumentParser(description="PBM Claims Rejection EDA")
# parser.add_argument("--file", required=True, help="Path to xlsx or csv dataset")
# args = parser.parse_args()

file_path = r"C:\Users\akila\Downloads\Book1.xlsx"  # replace with args.file for CLI usage

df = load_data(file_path)
df = preprocess(df)

section_overview(df)
section_age(df)
section_gender(df)
section_diagnosis(df)
section_drug(df)
section_drug_diag_combo(df)
section_service_type(df)
section_rejection_codes(df)
section_doctor(df)
section_facility(df)
section_correlation(df)
section_outliers(df)
section_time(df)
section_scores(df)
section_treatment_desc(df)
section_feature_importance(df)

print(f"\n✅  EDA complete. All outputs saved to: ./{OUT}/")
print("    Open the PNGs for charts, CSVs for tabular summaries.")



In [ ]:

# # ══════════════════════════════════════════════════════════════════════════════
# # MAIN
# # ══════════════════════════════════════════════════════════════════════════════

# def main():
#     parser = argparse.ArgumentParser(description="PBM Claims Rejection EDA")
#     parser.add_argument("--file", required=True, help="Path to xlsx or csv dataset")
#     args = parser.parse_args()

#     df = load_data(args.file)
#     df = preprocess(df)

#     section_overview(df)
#     section_age(df)
#     section_gender(df)
#     section_diagnosis(df)
#     section_drug(df)
#     section_drug_diag_combo(df)
#     section_service_type(df)
#     section_rejection_codes(df)
#     section_doctor(df)
#     section_facility(df)
#     section_correlation(df)
#     section_outliers(df)
#     section_time(df)
#     section_scores(df)
#     section_treatment_desc(df)
#     section_feature_importance(df)

#     print(f"\n✅  EDA complete. All outputs saved to: ./{OUT}/")
#     print("    Open the PNGs for charts, CSVs for tabular summaries.")


# if __name__ == "__main__":
#     main()